## Chunking, Vector Store & RAG with reduced Data Set 

In [83]:
import os

# data handling & viz
import pandas as pd
import matplotlib.pyplot as plt

# language preprocessing
import re #regex
from wordcloud import WordCloud
import spacy # DE stopwords

# langchain packages
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough #may not be needed due to QA removal
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
#from langchain_core.output_parsers import StrOutputParser appears to be buggy
from langchain_groq import ChatGroq
from langchain_classic import hub # alternative to 'from langchain import hub', because this gave errors


# environment variables
from dotenv import load_dotenv
load_dotenv()
import warnings
warnings.filterwarnings('ignore')

# instantiate ChatGroq with llama
llm = ChatGroq(
    model= "llama-3.1-8b-instant",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2
)


# refs for vector db handling
vector_db_name = "debates_lp19_basic"
vector_db_path = f"vector_databases/vector_db_{vector_db_name}"


In [84]:
# cleaned and reduced data set
df_exp_debates = pd.read_csv("data/debates_lp19.csv")

In [85]:
df_exp_debates.shape

(26869, 11)

In [86]:
# chunking function
# todo: chunks should be discourse-aware, dynamic
def chunk_documents(documents, chunk_size=200, chunk_overlap=50):
    """
    Splits documents into chunks of given size and overlap
    """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = text_splitter.split_documents(documents=documents)
    
    # add id to each chunk to map it later 
    for i, chunk in enumerate(chunks):
         chunk.metadata.update({
        "id": f"chunk_{i}",
    })
    
    return chunks

In [87]:
# convert df rows to Document objects, preserving metadata: store metadata with each chunk
documents = [
    Document(
        page_content=row['text'],
        metadata={
            'row_index': i, 
            'speaker_name': row['speech_identification_ent'], 
            'date': row['date'],
            'legislative_period': row['period'],
            'session': row['session'],
            'role': row['Role'],
            'governing_party': row['governing_Party'],
            'party': row['Party']
        }  
    )
    for i, row in df_exp_debates.iterrows() 
]



In [88]:
# chunking
chunks = chunk_documents(documents, chunk_size=500, chunk_overlap=50)

In [89]:
# display results of chunking function
print(f"number of chunks created: {len(chunks)}")

number of chunks created: 279370


In [90]:
# instantiate embedding model 
embedding = HuggingFaceEmbeddings(
        model_name='sentence-transformers/all-MiniLM-L6-v2',
        model_kwargs={'device': 'mps'}, # mps acceleration for M1 chip
        encode_kwargs={"normalize_embeddings": True}
    )

In [91]:
# function to create and save vector store 
def create_and_store(chunks,vector_db_path,embedding):
    """
    this function creates a vector store from chunks and saves it locally
    """
    # create the vector store 
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embedding, # parameter name is singular!
        distance_strategy=DistanceStrategy.COSINE  # or DistanceStrategy.DOT or DistanceStrategy.L2 
    )
    
     # save vector store locally
    vectorstore.save_local(vector_db_path)

    return vectorstore

In [92]:
# implement retrieval from FAISS db

def retrieve_from_vector_db(vector_db_path,embedding):
    """
    this function splits out a retriever object from a local vector store
    """
    
    vectorstore = FAISS.load_local(
        folder_path=vector_db_path,
        embeddings=embedding, # parameter name is plural!
        allow_dangerous_deserialization=True,
        distance_strategy=DistanceStrategy.COSINE  # or DistanceStrategy.DOT or DistanceStrategy.L2 
    )
    retriever = vectorstore.as_retriever(
        search_kwargs={
            'k':30, # k nearest
        }  
    )
    
    return retriever,vectorstore


In [93]:
# check if vector store exists. if no: creates vector store
if not os.path.exists(vector_db_path):
        print("Vector DB not found. Creating and embedding chunks.")
        all_embedding=create_and_store(chunks=chunks, vector_db_path=vector_db_path, embedding=embedding)
        print(f"Vector DB save to {vector_db_path}")
else:
    print(f"Vector DB found at {vector_db_path}. Skipping embedding.")

Vector DB found at vector_databases/vector_db_debates_lp19_basic. Skipping embedding.


In [94]:
# load the retriever and index
retriever,vectorstore = retrieve_from_vector_db(f"{vector_db_path}", embedding=embedding)


In [95]:
prompt = ChatPromptTemplate.from_template(
    """
    Du bist ein Assistent für Argumentationsanalyse, spezialisiert auf parlamentarische Debatten des Deutschen Bundestags.
    
    Deine Aufgabe ist es, die Argumentationsstruktur aus dem untenstehenden Redeauszug zu extrahieren.
    
    Definitionen:
    - claim: die zentrale politische Position oder der Vorschlag, für den oder gegen den argumentiert wird
    - supporting_premises: Aussagen, Fakten oder Gründe, die den claim stützen
    - opposing_premises: Gegenargumente, Einwände oder Positionen, die abgelehnt werden
    
    Regeln:
    - Verwende NUR Informationen aus dem Kontext, keine externen Kenntnisse
    - Zitiere oder paraphrasiere eng aus dem Text, nichts erfinden
    - Wenn kein klarer claim erkennbar ist, setze "claim" auf null
    - Wenn keine opposing_premises vorhanden sind, gib eine leere Liste zurück
    - Gib NUR valides JSON zurück, keine Erklärung, keine Einleitung
    
    Beispiel:
    Kontext: "Wir müssen die Energiewende beschleunigen. Der Klimawandel bedroht unsere Lebensgrundlagen. 
    Erneuerbare Energien schaffen zudem neue Arbeitsplätze. Die Opposition behauptet, dies sei zu teuer, 
    doch die Kosten des Nichtstuns sind weitaus höher."
    
    Ausgabe:
    {{
        "claim": "Die Energiewende muss beschleunigt werden",
        "supporting_premises": [
            "Der Klimawandel bedroht unsere Lebensgrundlagen",
            "Erneuerbare Energien schaffen neue Arbeitsplätze"
        ],
        "opposing_premises": [
            "Die Energiewende sei zu teuer"
        ],
        "confidence": "high"
        "reasoning": "Der Sprecher formuliert eine klare politische Forderung und stützt diese mit zwei expliziten Begründungen. Ein Gegenargument wird genannt und direkt entkräftet."
    }}
    
    Kontext:
    {context}
    """)

In [96]:
# runnable rag pipeline (LCEL)
def connect_chains(retriever):
    """
    Connects retriever, prompt and llm into a RAG chain. User input drives semantic search, but is not injected into the prompt.
    """
    rag_chain = (
        RunnableLambda(lambda x: {"context": retriever.invoke(x)})
        | prompt
        | llm
        | RunnableLambda(lambda msg: msg.content)
    )
    return rag_chain

In [97]:
rag_chain = connect_chains(retriever)

In [ ]:
# chat with rag loop
def chat_with_rag(chain):
    """
    Interactive function to chat with the RAG system.
    """
    print("Willkommen im ChatBundestag 🏛️. Gib eine Frage zu den Bundestagsdebatten ein. \nSchreibe 'exit', um den Chat zu beenden.\n")
    while True:
        user_input = input("Du: ")
        if user_input.lower() in ["exit", "quit"]:
            print("Chat wird beendet. Auf Wiedersehen!")
            break
        try:
            result = chain.invoke(user_input)
            print(f"ChatBundestag Antwort: {result}\n")
        except Exception as e:
            print(f" Fehler: {e}\n")

# run chat
chat_with_rag(rag_chain)

Willkommen im ChatBundestag 🏛️. Gib eine Frage zu den Bundestagsdebatten ein. 
Schreibe 'exit', um den Chat zu beenden.



Du:  Welche Argument sprechen für die Energiewendeß


ChatBundestag Antwort: Nachdem ich den Text analysiert habe, kann ich folgendes Ergebnis ermitteln:

{
  "claim": "Die Energiewende muss beschleunigt werden",
  "supporting_premises": [
    "Der Klimawandel bedroht unsere Lebensgrundlagen",
    "Erneuerbare Energien schaffen neue Arbeitsplätze"
  ],
  "opposing_premises": [
    "Die Energiewende sei zu teuer"
  ],
  "confidence": "high",
  "reasoning": "Der Sprecher formuliert eine klare politische Forderung und stützt diese mit zwei expliziten Begründungen. Ein Gegenargument wird genannt und direkt entkräftet."
}

Ich habe folgende Argumentationsstruktur identifiziert:

* Der Sprecher formuliert eine klare politische Forderung, nämlich die Beschleunigung der Energiewende.
* Zwei explizite Begründungen werden genannt, um diese Forderung zu stützen:
 + Der Klimawandel bedroht unsere Lebensgrundlagen.
 + Erneuerbare Energien schaffen neue Arbeitsplätze.
* Ein Gegenargument wird genannt, nämlich dass die Energiewende zu teuer sei.
* Diese